# 04 — Temporal Audit & Candidate t0

**Objetivo:** reconstruir la semántica temporal y evaluar la viabilidad del timestamp de decisión.

Este notebook es un **laboratorio reproducible**. La lógica reutilizable vive en `src/argos/`;
aquí solo se parametriza, ejecuta, inspecciona evidencia y documentan decisiones.

## Preguntas
- ¿Qué fechas existen y qué cobertura tienen?
- ¿Qué rangos temporales cubre cada fuente?
- ¿Qué campos no parsean correctamente?
- ¿Se cumplen órdenes lógicos básicos de fechas?
- ¿`fecha_de_firma` es un `t0` operativo razonable o debe cambiarse?

Este notebook es crítico: una columna presente hoy no necesariamente existía en el momento histórico de predicción.

In [ ]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths
from argos.config.registry import load_registry, discover_dataset_files

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
DATASETS, RELATIONSHIPS = load_registry(paths.configs / "datasets.toml")
FILES = discover_dataset_files(paths.bronze, DATASETS)

with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

{k: p.name for k, p in FILES.items()}

In [ ]:
from argos.profiling.temporal import audit_all_temporal, audit_order_rule
from argos.common.reports import write_report

cfg = PROJECT_CONFIG["profiling"]
temporal_profile = audit_all_temporal(
    FILES,
    sample_size=cfg["csv_sample_size"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

write_report(
    temporal_profile,
    paths.reports / "profiling" / "temporal_profile.parquet",
)

temporal_profile.sort_values(["dataset", "column"])

In [ ]:
# Reglas de consistencia temporal conocidas en Contratos. Si una columna no existe,
# la regla queda registrada como missing_columns en lugar de fallar silenciosamente.
rules = [
    ("fecha_de_firma", "fecha_de_inicio_del_contrato"),
    ("fecha_de_inicio_del_contrato", "fecha_de_fin_del_contrato"),
    ("fecha_inicio_liquidacion", "fecha_fin_liquidacion"),
]

temporal_rules = pd.DataFrame([
    audit_order_rule(
        "contracts",
        FILES["contracts"],
        earlier,
        later,
        sample_size=cfg["csv_sample_size"],
        temp_directory=paths.root / ".cache" / "duckdb",
    )
    for earlier, later in rules
])

write_report(
    temporal_rules,
    paths.reports / "profiling" / "temporal_rules.parquet",
)

temporal_rules

In [ ]:
# Evidencia para el candidato inicial t0 = fecha_de_firma.
t0_candidates = temporal_profile.loc[
    (temporal_profile["dataset"] == "contracts")
    & temporal_profile["normalized_column"].isin([
        "fecha_de_firma",
        "fecha_de_inicio_del_contrato",
    ]),
    ["column", "coverage", "parse_success_rate_nonnull", "min_timestamp", "max_timestamp"]
]

t0_candidates

## Decisión
No congeles `t0` por intuición. Documenta por qué el timestamp elegido representa el instante de decisión y qué información era realmente observable entonces.